# OpenHPL AGC_SMIB on Railway
This notebook compares a reduced four-state LFC model against the nonlinear OpenHPL hydraulic simulation for a 50% to 60% load step at t=5 s. The reduced model uses zero load-frequency damping to match the nonlinear Grid configuration (`mu=0`, `Lambda=0`).


In [ ]:
import pathlib, subprocess, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
RES=pathlib.Path('/workspace/results/AGC_SMIB_res.csv')
print(subprocess.check_output(['omc','--version'], text=True).strip())
if not RES.exists(): subprocess.run(['/workspace/OpenHPL-repo/railway/run_model.sh'], check=True)
df=pd.read_csv(RES)
print('Columns:', df.columns.tolist())


In [ ]:
PBASE=2.5e6; F0=50.; H=3.; D=0.; R=.05; TG=.30; TT=.50; KI=1.50; STEP=5.; DPL=.10
def rhs(t,x):
    dfp,dg,dpm,xi=x; dl=0. if t<STEP else DPL; ef=-dfp
    return [(dpm-dl-D*dfp)/(2*H),((ef/R+KI*xi)-dg)/TG,(dg-dpm)/TT,ef]
sol=solve_ivp(rhs,(0,65),[0,0,0,0],max_step=.01,rtol=1e-9,atol=1e-11,dense_output=True)
ta=np.linspace(0,65,6501); xa=sol.sol(ta); fa=F0*(1+xa[0]); pma=(.5+xa[2])*PBASE/1e6
freq_col='frequency_Hz' if 'frequency_Hz' in df else next(c for c in df.columns if 'generator.f' in c)
if freq_col!='frequency_Hz': df['frequency_Hz']=50*df[freq_col]
pm_col='mechanicalPower' if 'mechanicalPower' in df else next(c for c in df.columns if 'turbine.Wdot_s' in c)
df['Pm_MW']=df[pm_col]/1e6
load_col='electricalLoad' if 'electricalLoad' in df else next(c for c in df.columns if 'loadStep.y' in c)
df['Load_MW']=df[load_col]/1e6
gate_col='guideVane' if 'guideVane' in df else next(c for c in df.columns if 'governor.gate' in c)
flow_col='turbineFlow' if 'turbineFlow' in df else next(c for c in df.columns if 'turbine.Vdot' in c)
post=df['time']>=5
i=df.loc[post,'frequency_Hz'].idxmin()
nadir=float(df.loc[i,'frequency_Hz']); tn=float(df.loc[i,'time'])
ia=np.where(ta>=5)[0][np.argmin(fa[ta>=5])]
print('COMPARISON_START')
print(f'OpenHPL_nadir_Hz={nadir:.9f}')
print(f'OpenHPL_nadir_time_s={tn:.6f}')
print(f'OpenHPL_f65_Hz={df.iloc[-1]["frequency_Hz"]:.9f}')
print(f'OpenHPL_Pm65_MW={df.iloc[-1]["Pm_MW"]:.9f}')
print(f'Reduced_nadir_Hz={fa[ia]:.9f}')
print(f'Reduced_nadir_time_s={ta[ia]:.6f}')
print(f'Reduced_f65_Hz={fa[-1]:.9f}')
print(f'Reduced_Pm65_MW={pma[-1]:.9f}')
print(f'Nadir_difference_Hz={nadir-fa[ia]:.9f}')
print('COMPARISON_END')


In [ ]:
plt.figure(figsize=(9,5)); plt.plot(df['time'],df['frequency_Hz'],label='OpenHPL nonlinear'); plt.plot(ta,fa,'--',label='Reduced model'); plt.axvline(5,linestyle=':'); plt.xlabel('Time [s]'); plt.ylabel('Frequency [Hz]'); plt.title('AGC_SMIB frequency: nonlinear vs reduced'); plt.grid(True); plt.legend(); plt.tight_layout(); plt.savefig('/workspace/results/agc_frequency_compare.png',dpi=180); plt.show()
plt.figure(figsize=(9,5)); plt.plot(df['time'],df['Pm_MW'],label='OpenHPL turbine power'); plt.plot(df['time'],df['Load_MW'],label='Electrical load'); plt.plot(ta,pma,'--',label='Reduced mechanical power'); plt.axvline(5,linestyle=':'); plt.xlabel('Time [s]'); plt.ylabel('Power [MW]'); plt.title('AGC_SMIB power balance'); plt.grid(True); plt.legend(); plt.tight_layout(); plt.savefig('/workspace/results/agc_power_compare.png',dpi=180); plt.show()
plt.figure(figsize=(9,5)); plt.plot(df['time'],df[gate_col],label='Guide vane'); plt.plot(df['time'],df[flow_col],label='Flow [m3/s]'); plt.axvline(5,linestyle=':'); plt.xlabel('Time [s]'); plt.title('Governor and hydraulic response'); plt.grid(True); plt.legend(); plt.tight_layout(); plt.savefig('/workspace/results/agc_gate_flow.png',dpi=180); plt.show()
df[['time','frequency_Hz','Pm_MW','Load_MW',gate_col,flow_col]].to_csv('/workspace/results/agc_smib_results.csv',index=False)
print('AGC_DATA_START')
for ts in [0,1,4.9,5,5.1,5.2,5.4,5.5,5.6,5.7,5.8,6,7,10,20,40,65]:
    j=(df['time']-ts).abs().idxmin(); r=df.loc[j]
    print(f"{r['time']:.4f},{r['frequency_Hz']:.9f},{r['Pm_MW']:.9f},{r['Load_MW']:.9f},{r[gate_col]:.9f},{r[flow_col]:.9f}")
print('AGC_DATA_END')
